# Indexing TREC Robust 2005 by OpenSearch for BM25 Model

- [aquaint/trec-robust-2005](https://ir-datasets.com/aquaint.html#aquaint/trec-robust-2005)

In [1]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv beautifulsoup4

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Index a Corpus for BM25 Model

In [4]:
import ir_datasets
dataset_name = "aquaint/trec-robust-2005"
dataset = ir_datasets.load(dataset_name)

In [5]:
index_name = "trec_robust_2005_bm25"

In [6]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

{'acknowledged': True}


In [7]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True,
 'index': 'trec_robust_2005_bm25',
 'shards_acknowledged': True}


Indexing

In [8]:
from bs4 import BeautifulSoup
def parse_marked_up_doc(marked_up_doc):
    # Parse the content using BeautifulSoup
    soup = BeautifulSoup(marked_up_doc, 'html.parser')

    # Extract the title from the <HEADLINE> tag (empty string if absent)
    headline_tag = soup.find('headline')
    title = headline_tag.get_text(strip=True) if headline_tag else ""

    # Extract the body text by joining the <P> paragraph tags
    text = ' '.join(p.get_text(strip=True) for p in soup.find_all('p'))

    return title, text

In [9]:
def prepare_documents(dataset):
    """
    Prepare individual documents for indexing, with progress tracking.
    AQUAINT wraps the title in <HEADLINE> and the body in <P> tags, so we
    parse `marked_up_doc` to keep the title as a separate field. Docs with
    no <HEADLINE> get an empty title.
    """
    total_docs = sum(1 for _ in dataset.docs_iter())  # Count the total documents
    progress = tqdm(total=total_docs, desc="Indexing Documents")  # Progress bar

    for doc in dataset.docs_iter():
        title, text = parse_marked_up_doc(doc.marked_up_doc)
        text = text.replace("\n", " ")
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "title": title,
                "text": text
            }
        }
        progress.update(1)  # Update progress bar

    progress.close()  # Close the progress bar

In [10]:
from opensearchpy.helpers import bulk
success, failed = bulk(client, prepare_documents(dataset), index=index_name)
print(f"indexed: {success},  failed: {len(failed)}")

Indexing Documents: 100%|██████████| 1033461/1033461 [07:07<00:00, 2419.53it/s]

indexed: 1033461,  failed: 0


---
### (Optional) Re-index documents missing from the first pass

BM25 indexing has no server-side pipeline, so this is just a completeness
check: diff the corpus against what's in the index and re-index anything
missing.

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast)
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index any docs the scan diff flagged as missing, printing every error.
from opensearchpy.helpers import streaming_bulk

def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        title, text = parse_marked_up_doc(docstore.get(doc_id).marked_up_doc)
        text = text.replace("\n", " ")
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "title": title, "text": text},
        }

retry_ok, retry_failed = 0, []
for ok, item in streaming_bulk(
    client,
    prepare_missing(dataset, missing),
    index=index_name,
    chunk_size=500,
    request_timeout=300,
    raise_on_error=False,
    raise_on_exception=False,
):
    retry_ok += ok
    if not ok:
        retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed:          # full error detail for every failure
    pprint.pprint(item)
    print("-" * 80)